In [2]:
import pandas as pd
import numpy as np
import os

def standardize(score): # See no majority as contentious
    if score == 0.5:
        return 1
    else:
        return score

def df_standardize(df):
    df['score'] = df['score'].apply(standardize)
    df['output_score'] = df['output_score'].apply(standardize)
    return df

def calculate_accuracy(data):
    data_h = data[data['extract_id'].str.startswith('H')]  # experts annotated samples
    data_noth = data[~data['extract_id'].str.startswith('H')] # crowd workers annotated samples
    num_data_h = len(data_h)
    num_data_noth = len(data_noth)
    num_data = len(data)
    accuracy_h = (data_h['output_score'] == data_h['score']).mean() 
    accuracy_noth = (data_noth['output_score'] == data_noth['score']).mean()
    accuracy_all = (data['output_score'] == data['score']).mean()
    return accuracy_h, accuracy_noth, accuracy_all, num_data_h, num_data_noth, num_data

def accuracy_moe(accuracy, n, confidence=0.95):
    z = 1.96 if confidence == 0.95 else 1.645
    moe = z * np.sqrt(accuracy * (1 - accuracy) / n)
    return moe

input_dir = "./gpt and llama"

for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_dir, filename)
        data = pd.read_csv(filepath)
        data = df_standardize(data)
        accuracy_experts, accuracy_crowd, accuracy_all, num_experts, num_crowd, num_all = calculate_accuracy(data)
        print(f"Dataset: {filename}")
        print(f"Accuracy based on experts: {accuracy_experts}, Accuracy based on crowd workers: {accuracy_crowd}, Overall Accuracy: {accuracy_all}")
        moe_experts = accuracy_moe(accuracy_experts, num_experts)
        moe_crowd = accuracy_moe(accuracy_crowd, num_crowd)
        moe_all = accuracy_moe(accuracy_all, num_all)
        print(f"Margin of Error for experts: {moe_experts}, Margin of Error for crowd workers: {moe_crowd}, Margin of Error for overall: {moe_all}")

Dataset: gpt-4o  ori_en_CandR.csv
Accuracy based on experts: 0.8629032258064516, Accuracy based on crowd workers: 0.8336724313326551, Overall Accuracy: 0.8354066985645933
Margin of Error for experts: 0.06053964823229229, Margin of Error for crowd workers: 0.016460542871858653, Margin of Error for overall: 0.015897834924321085
Dataset: llama3-70b simp_nl_CandR.csv
Accuracy based on experts: 0.8306451612903226, Accuracy based on crowd workers: 0.8255340793489319, Overall Accuracy: 0.8258373205741627
Margin of Error for experts: 0.06601639477056873, Margin of Error for crowd workers: 0.016775949922113335, Margin of Error for overall: 0.016259520342061372
Dataset: gpt-4o  simp_en_C.csv
Accuracy based on experts: 0.8306451612903226, Accuracy based on crowd workers: 0.7502543234994914, Overall Accuracy: 0.755023923444976
Margin of Error for experts: 0.06601639477056873, Margin of Error for crowd workers: 0.019134519345752464, Margin of Error for overall: 0.018438480235626747
Dataset: llama3-

In [3]:
# Accuracy for consensus samples

for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_dir, filename)
        data = pd.read_csv(filepath)
        data = df_standardize(data)
        consensus_data = pd.read_csv("ready_annotations_1254.csv")
        extract_ids_consensus = consensus_data['extract_id'].unique()
        filtered_data = data[data['extract_id'].isin(extract_ids_consensus)]
        accuracy_experts, accuracy_crowd, accuracy_all, num_experts, num_crowd, num_all = calculate_accuracy(filtered_data)
        print(f"Dataset: {filename}")
        print(f"Accuracy based on experts: {accuracy_experts}, Accuracy based on crowd workers: {accuracy_crowd}, Overall Accuracy: {accuracy_all}")
        moe_experts = accuracy_moe(accuracy_experts, num_experts)
        moe_crowd = accuracy_moe(accuracy_crowd, num_crowd)
        moe_all = accuracy_moe(accuracy_all, num_all)
        print(f"Margin of Error for experts: {moe_experts}, Margin of Error for crowd workers: {moe_crowd}, Margin of Error for overall: {moe_all}")

Dataset: gpt-4o  ori_en_CandR.csv
Accuracy based on experts: 0.9863013698630136, Accuracy based on crowd workers: 0.9170194750211685, Overall Accuracy: 0.9210526315789473
Margin of Error for experts: 0.026664781506750317, Margin of Error for crowd workers: 0.015732889452558504, Margin of Error for overall: 0.01492513802258816
Dataset: llama3-70b simp_nl_CandR.csv
Accuracy based on experts: 0.9452054794520548, Accuracy based on crowd workers: 0.9187129551227773, Overall Accuracy: 0.9202551834130781
Margin of Error for experts: 0.05220670961741541, Margin of Error for crowd workers: 0.015585893722987377, Margin of Error for overall: 0.014993833055971023
Dataset: gpt-4o  simp_en_C.csv
Accuracy based on experts: 0.9178082191780822, Accuracy based on crowd workers: 0.8272650296359018, Overall Accuracy: 0.8325358851674641
Margin of Error for experts: 0.06300642055760598, Margin of Error for crowd workers: 0.021559743541686473, Margin of Error for overall: 0.02066662995522431
Dataset: llama3-